In [1]:
import pandas as pd
import pyarrow as pa
import numpy as np
import openpyxl

## First thing first let's figure out gene names. I ran an all-vs-all blast to line them up between the RBTnSeq reference and the dubseq reference:

In [ ]:
homologs = pd.read_csv('../../../Moriniere_2026_Metadata/recip_best_hits_blast/keio_homolog_hits.tsv', sep='\t')
locusId_map = {i[0]:i[1] for i in np.array(homologs[['query', 'best_hit']])}

In [ ]:
def get_gene_name(row):
    if pd.notnull(row['name']):
        return row['name']
    else:
        return row['original_name']
    
def get_desc(row):
    if pd.notnull(row['desc']):
        return row['desc']
    else:
        return row['product']

rb_gene_table = pd.read_csv('../barseq_browser/data/Keio/genes_w_ecocyc.tab', sep='\t')
dubseq_data = pd.read_csv('../../../Moriniere_2026_Metadata/FEBA_BS_455-Dubseq/gscore/gscore_base.tsv', sep='\t').rename(columns={
    'name': 'original_name', 'pos_from': 'begin', 'pos_to': 'end', 'contig_id': 'scaffoldId'
    })
dubseq_data['locusId'] = dubseq_data['locus_tag'].apply(lambda x: locusId_map.get(x, ''))

dubseq_ref_genes = dubseq_data[['locus_tag', 'scaffoldId', 'begin', 'end', 'strand', 'original_name', 'gene_type', 'locusId', 'product']].merge(rb_gene_table[['locusId', 'sysName', 'name', 'desc', 'ecocyc_id']], on='locusId', how='left').rename(columns={'locus_tag': 'locusId', 'locusId': 'rbtnseq_ref_locusId'})
dubseq_ref_genes['rbtnseq_ref_locusId'] = dubseq_ref_genes['rbtnseq_ref_locusId'].fillna('NA').apply(lambda x: str(x))
dubseq_ref_genes['name'] = dubseq_ref_genes.apply(get_gene_name, axis=1)
dubseq_ref_genes['desc'] = dubseq_ref_genes.apply(get_desc, axis=1)
dubseq_ref_genes.to_csv('../barseq_browser/data/Keio/genes_w_ecocyc_dubseq_ref.tab', sep='\t', index=False)
dubseq_ref_genes

,locusId,scaffoldId,begin,end,strand,original_name,gene_type,rbtnseq_ref_locusId,product,sysName,name,desc,ecocyc_id
0,BW25113_0001,CP009273.1,190,255,+,thrL,CDS,14146,thr operon leader peptide,b0001,thrL,thr operon leader peptide (NCBI),EG11277
1,BW25113_0002,CP009273.1,337,2799,+,thrA,CDS,14147,Bifunctional aspartokinase/homoserine dehydrog...,b0002,thrA,bifunctional aspartokinase I/homeserine dehydr...,EG10998
2,BW25113_0003,CP009273.1,2801,3733,+,thrB,CDS,14148,homoserine kinase,b0003,thrB,homoserine kinase (NCBI),EG10999
3,BW25113_0004,CP009273.1,3734,5020,+,thrC,CDS,14149,L-threonine synthase,b0004,thrC,threonine synthase (NCBI),EG11000
4,BW25113_0005,CP009273.1,5234,5530,+,yaaX,CDS,14150,DUF2502 family putative periplasmic protein,b0005,yaaX,hypothetical protein (NCBI),G6081
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4504,BW25113_4399,CP009273.1,4626513,4627937,+,creC,CDS,,sensory histidine kinase in two-component regu...,NaN,creC,sensory histidine kinase in two-component regu...,NaN
4505,BW25113_4400,CP009273.1,4627995,4629347,+,creD,CDS,,inner membrane protein,NaN,creD,inner membrane protein,NaN
4506,BW25113_4401,CP009273.1,4629407,4630123,-,arcA,CDS,,response regulator in two-component regulatory...,NaN,arcA,response regulator in two-component regulatory...,NaN
4507,BW25113_4402,CP009273.1,4630219,4630359,+,yjjY,CDS,,uncharacterized protein,NaN,yjjY,uncharacterized protein,NaN


## First up, getting gene level data. This should mirror RBTnSeq.feather, so locusID, sysName, desc, then scores and any filtering, by set number:

In [ ]:
from collections import defaultdict
from glob import glob

flist = glob('../../../Moriniere_2026_Metadata/Dubseq_data/FEBA_BS_*-Dubseq/gscore/*.gscore.tsv')
locusId_to_scores = defaultdict(dict)
setNames = []
for f in flist:
    setName = f.split('/')[-1].split('.')[0]
    setNames.append(setName)
    for row in np.array(pd.read_csv(f, sep='\t')[['locus_tag', 'score_cnnls']]):
        locusId = row[0] #sys_to_locusId.get('b'+row[0].split('_')[-1], 'b'+row[0].split('_')[-1])
        locusId_to_scores[locusId][setName] = row[1]
    
mat = []
for locusId in locusId_to_scores:
    td = locusId_to_scores[locusId]
    mat.append([locusId]+[td.get(i, np.nan) for i in setNames])
    
dubseq_data = dubseq_ref_genes.merge(pd.DataFrame(mat, columns=['locusId']+setNames), on='locusId', how='left')

schema = pa.Schema.from_pandas(dubseq_data, preserve_index=False)
table = pa.Table.from_pandas(dubseq_data, preserve_index=False)
sink = '../barseq_browser/data/Keio/DubSeq.feather'
writer = pa.ipc.new_file(sink, schema)
writer.write(table)
writer.close()

dubseq_data

,locusId,scaffoldId,begin,end,strand,original_name,gene_type,rbtnseq_ref_locusId,product,sysName,...,S33,S66,S51,S40,S77,S22,S15,S4,S88,S94
0,BW25113_0001,CP009273.1,190,255,+,thrL,CDS,14146,thr operon leader peptide,b0001,...,0.269021,0.184058,0.006816,0.269021,0.178823,0.386342,0.178823,0.224627,-0.027446,0.224627
1,BW25113_0002,CP009273.1,337,2799,+,thrA,CDS,14147,Bifunctional aspartokinase/homoserine dehydrog...,b0002,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,BW25113_0003,CP009273.1,2801,3733,+,thrB,CDS,14148,homoserine kinase,b0003,...,0.000000,0.289312,0.342005,0.000000,0.000000,-0.036503,0.000000,0.000000,0.430084,0.084443
3,BW25113_0004,CP009273.1,3734,5020,+,thrC,CDS,14149,L-threonine synthase,b0004,...,0.689253,0.441982,-0.069379,0.689253,0.601069,1.268640,0.655429,0.672606,0.072497,0.646217
4,BW25113_0005,CP009273.1,5234,5530,+,yaaX,CDS,14150,DUF2502 family putative periplasmic protein,b0005,...,0.381820,0.230627,-0.101612,0.381820,0.378801,0.000000,0.297260,0.340201,0.397213,0.337562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4504,BW25113_4399,CP009273.1,4626513,4627937,+,creC,CDS,,sensory histidine kinase in two-component regu...,NaN,...,0.000000,-1.538336,0.000000,-0.953374,-0.843572,-0.688263,-0.843572,-0.997768,0.007618,-0.994514
4505,BW25113_4400,CP009273.1,4627995,4629347,+,creD,CDS,,inner membrane protein,NaN,...,-1.240212,-1.428934,0.018961,-0.990212,-1.080410,0.095726,0.569568,-1.284606,-0.494197,-1.284606
4506,BW25113_4401,CP009273.1,4629407,4630123,-,arcA,CDS,,response regulator in two-component regulatory...,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4507,BW25113_4402,CP009273.1,4630219,4630359,+,yjjY,CDS,,uncharacterized protein,NaN,...,-1.989029,0.747937,-0.847556,-0.989029,-1.079227,-1.079227,-2.079227,-2.033423,-0.463567,-2.033423


## Making the assay sheet, should match RBTnseq_sets.csv

In [ ]:
meta = pd.read_excel('../../../Moriniere_2026_Metadata/Dubseq_sets_v2.xlsx')
from collections import Counter
phage_reps = []
phage_rep_d = {'Keio': Counter(), 'BL21': Counter()}
for idx, row in meta.iterrows():
    nickname = row['nickname']
    phage_rep_d[nickname][row['phage']] += 1
    phage_reps.append(row['phage']+' R'+str(phage_rep_d[nickname][row['phage']]))

meta['phage_rep'] = phage_reps
meta.to_csv('../barseq_browser/data/Dubseq_sets.csv', index=False)
meta

,host,strain,nickname,library,phage,set,time0,no_phage_control,phage_rep
0,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,Time0,S377,S377;S378;S383;S384,NaN,Time0 R1
1,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,Time0,S378,S377;S378;S383;S384,NaN,Time0 R2
2,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,Time0,S383,S377;S378;S383;S384,NaN,Time0 R3
3,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,Time0,S384,S377;S378;S383;S384,NaN,Time0 R4
4,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,Time0,S473,S473;S474;S479,S480,Time0 R5
...,...,...,...,...,...,...,...,...,...
298,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,TP6,S28,S90;S95;S96,S89,TP6 R2
299,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,TulB,S495,S569;S570;S575,S576,TulB R1
300,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,TulB,S496,S569;S570;S575,S576,TulB R2
301,Escherichia coli,BW25113,Keio,BW25113 dubseq in BW25113,phi92,S407,S473;S474;S479,S480,phi92 R1


## The fragment counts and scores are the only other thing, to be served by the server

In [ ]:
BC_to_score_and_count = defaultdict(dict)
setNames = []
for data_dir in ['455', '478']:
    flist = glob(f'../../../Moriniere_2026_Metadata/Dubseq_data/FEBA_BS_{data_dir}-Dubseq/gscore/*.fscore.tsv')
    for f in flist:
        setName = f.split('/')[-1].split('.')[0]
        setNames.append(setName)
        for row in np.array(pd.read_csv(f, sep='\t')[['barcode', 'score', 'stress_read_count']]):
            BC_to_score_and_count[row[0]][setName] = row[1:]
    
mat1 = []
mat2 = []
for barcode in BC_to_score_and_count:
    td = BC_to_score_and_count[barcode]
    mat1.append([barcode]+[td.get(i, [np.nan, 0])[0] for i in setNames])
    mat2.append([barcode]+[td.get(i, [np.nan, 0])[1] for i in setNames])
    
score_data = pd.DataFrame(mat1, columns=['barcode']+setNames)
count_data = pd.DataFrame(mat2, columns=['barcode']+setNames)

dubseq_base_data = pd.concat([pd.read_csv(f'../../../Moriniere_2026_Metadata/Dubseq_data/FEBA_BS_{data_dir}-Dubseq/gscore/fscore_base.tsv', sep='\t')[['barcode', 'contig_id', 'pos_from', 'pos_to']] for data_dir in ['455', '478']]).drop_duplicates().sort_values(by='pos_from')

score_done = dubseq_base_data.merge(score_data, on='barcode', how='left')
count_done = dubseq_base_data.merge(count_data, on='barcode', how='left')

score_done = score_done.rename(columns={'pos_from': 'begin', 'pos_to': 'end', 'contig_id': 'scaffold'})
count_done = count_done.rename(columns={'pos_from': 'begin', 'pos_to': 'end', 'contig_id': 'scaffold'})

score_done.sort_values(by='begin').to_parquet('../../../PhageDataServer/data/Keio/DubSeq_scores.parquet')
count_done.sort_values(by='begin').to_parquet('../../../PhageDataServer/data/Keio/DubSeq_counts.parquet')

In [135]:
missing = []
for i in set(meta['set']):
    if i not in list(count_done.columns):
        missing.append(i)
print(sorted(missing))

[]
